In [1]:
import pandas as pd
df = pd.read_csv("Dataset/Diabetes_data_og.csv")

df = df.drop_duplicates()
df

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,2.0,1.0,1.0,1.0,18.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif

# ---- CONFIG ----
FILE = "Dataset/Diabetes_data_og.csv"   # change to your filename (.xlsx or .csv)
SHEET = 0
TARGET = "Diabetes_012"
SAMPLE = 60000           # rows used for speed; set to None for all rows
# ----------------------

df = pd.read_csv(FILE) if FILE.endswith(".csv") else pd.read_excel(FILE, sheet_name=SHEET)
df = df.dropna(axis=1, how="all").dropna()

if SAMPLE and len(df) > SAMPLE:
    df = df.sample(SAMPLE, random_state=42)

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=5, class_weight="balanced",
    n_jobs=-1, random_state=42
).fit(X_train, y_train)
print(f"Test accuracy: {rf.score(X_test, y_test):.3f}\n")

perm = permutation_importance(
    rf, X_test, y_test, n_repeats=5, scoring="balanced_accuracy",
    n_jobs=-1, random_state=42
)

result = pd.DataFrame({
    "feature": X.columns,
    "rf_importance": rf.feature_importances_,
    "permutation_importance": perm.importances_mean,
    "mutual_info": mutual_info_classif(X, y, random_state=42),
    "corr_with_target": X.corrwith(y).values,
}).sort_values("permutation_importance", ascending=False)

print(result.round(4).to_string(index=False))
result.to_csv("feature_importance_results.csv", index=False)

result.sort_values("permutation_importance").plot.barh(
    x="feature", y="permutation_importance", legend=False, figsize=(8, 7)
)
plt.title("Feature importance (permutation, balanced accuracy)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

Test accuracy: 0.719

             feature  rf_importance  permutation_importance  mutual_info  corr_with_target
             GenHlth         0.0997                  0.0295       0.0486            0.3022
                 BMI         0.1635                  0.0207       0.0302            0.2201
                 Age         0.1357                  0.0139       0.0243            0.1878
              HighBP         0.0655                  0.0127       0.0460            0.2753
            HighChol         0.0451                  0.0087       0.0295            0.2181
                 Sex         0.0305                  0.0078       0.0046            0.0282
HeartDiseaseorAttack         0.0166                  0.0064       0.0126            0.1727
            DiffWalk         0.0233                  0.0043       0.0214            0.2163
            PhysHlth         0.0723                  0.0020       0.0149            0.1764
           Education         0.0561                  0.0018       0.

ValueError: No engine for filetype: 'csv'